In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB

RANDOM_STATE = 42

In [2]:
train_df = pd.read_csv("C:/Users/ahmed/Downloads/aspect_extraction_and_classification/amazon_train_english.csv", header=None, names=["label", "title", "text"])
test_df  = pd.read_csv("C:/Users/ahmed/Downloads/aspect_extraction_and_classification/amazon_test_english.csv", header=None, names=["label", "title", "text"])

X_train = train_df["text"]
y_train = train_df["label"]

X_test = test_df["text"]
y_test = test_df["label"]

print(f"Loaded {len(X_train)} training samples and {len(X_test)} test samples.")


# =======================================================
# 2. LABEL NORMALIZATION (IMPORTANT)
# =======================================================

unique_labels = sorted(y_train.unique())
label_map = {old: new for new, old in enumerate(unique_labels)}

print("Label mapping:", label_map)

y_train = y_train.map(label_map)
y_test  = y_test.map(label_map)





Loaded 3580535 training samples and 397826 test samples.
Label mapping: {1: 0, 2: 1}


In [3]:
print("\n[+] Fitting TF-IDF...")
vectorizer = TfidfVectorizer(stop_words="english", max_features=50000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf  = vectorizer.transform(X_test)

print("TF-IDF shape:", X_train_tfidf.shape)


[+] Fitting TF-IDF...
TF-IDF shape: (3580535, 50000)


In [4]:
models = {
    "LogReg": LogisticRegression(max_iter=2000, n_jobs=-1),
    "LinearSVM": LinearSVC(),
    "MultinomialNB": MultinomialNB()
}



In [5]:
# Use fewer splits for 3M samples to keep runtime sane
kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
results = {}

print("\n================ MODEL COMPARISON (Stratified K-Fold) ================\n")

for name, model in models.items():
    print(f"\n===== Evaluating {name} =====")
    fold_acc = []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train_tfidf, y_train), start=1):
        print(f"  Fold {fold_idx}...")

        Xtr = X_train_tfidf[train_idx]
        Xval = X_train_tfidf[val_idx]
        ytr = y_train.iloc[train_idx]
        yval = y_train.iloc[val_idx]

        model.fit(Xtr, ytr)
        preds = model.predict(Xval)

        acc = accuracy_score(yval, preds)
        fold_acc.append(acc)
        print(f"    Fold {fold_idx} accuracy: {acc:.4f}")

    mean_acc = np.mean(fold_acc)
    results[name] = mean_acc

    print(f"--> {name} Fold Accuracies: {fold_acc}")
    print(f"--> {name} Mean Accuracy: {mean_acc:.4f}")


# =======================================================
# 4. SELECT BEST MODEL FOR GRID SEARCH
# =======================================================

best_model_name = max(results, key=results.get)
print("\n================ BEST MODEL FROM K-FOLD =================")
print(f"Best model: {best_model_name} with mean CV accuracy = {results[best_model_name]:.4f}\n")



================ MODEL COMPARISON (Stratified K-Fold) ================


===== Evaluating LogReg =====
  Fold 1...
    Fold 1 accuracy: 0.8751
  Fold 2...
    Fold 2 accuracy: 0.8740
  Fold 3...
    Fold 3 accuracy: 0.8748
--> LogReg Fold Accuracies: [0.8751449503649733, 0.8740205377072036, 0.8748331603144002]
--> LogReg Mean Accuracy: 0.8747

===== Evaluating LinearSVM =====
  Fold 1...
    Fold 1 accuracy: 0.8752
  Fold 2...
    Fold 2 accuracy: 0.8754
  Fold 3...
    Fold 3 accuracy: 0.8753
--> LinearSVM Fold Accuracies: [0.8751968978946169, 0.8753560919370731, 0.8752772282785831]
--> LinearSVM Mean Accuracy: 0.8753

===== Evaluating MultinomialNB =====
  Fold 1...
    Fold 1 accuracy: 0.8204
  Fold 2...
    Fold 2 accuracy: 0.8204
  Fold 3...
    Fold 3 accuracy: 0.8207
--> MultinomialNB Fold Accuracies: [0.8203872269403241, 0.8203528745416887, 0.8206702745094097]
--> MultinomialNB Mean Accuracy: 0.8205

================ BEST MODEL FROM K-FOLD =================
Best model: LinearS

In [6]:
if best_model_name == "LogReg":
    base_model = LogisticRegression(max_iter=2000, n_jobs=-1)


elif best_model_name == "LinearSVM":
    base_model = LinearSVC()
 

elif best_model_name == "MultinomialNB":
    base_model = MultinomialNB()





In [7]:
base_model.fit(X_train_tfidf, y_train)

,penalty,'l2'
,loss,'squared_hinge'
,dual,'auto'
,tol,0.0001
,C,1.0
,multi_class,'ovr'
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,verbose,0
,random_state,None


In [8]:
print("\n================ FINAL TEST RESULTS ================\n")

test_preds = base_model.predict(X_test_tfidf)

print("Test Accuracy:", accuracy_score(y_test, test_preds))
print("\nClassification Report:")
print(classification_report(y_test, test_preds))


================ FINAL TEST RESULTS ================

Test Accuracy: 0.8766822681272717

Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.87      0.88    199086
           1       0.87      0.88      0.88    198740

    accuracy                           0.88    397826
   macro avg       0.88      0.88      0.88    397826
weighted avg       0.88      0.88      0.88    397826



In [9]:
import joblib

# Save the trained final model
joblib.dump(base_model, "english_polarity/english_polarity_model.pkl")

# Save the TF-IDF vectorizer
joblib.dump(vectorizer, "english_polarity/english_polarity_vectorizer.pkl")

print("Model and vectorizer saved successfully!")

Model and vectorizer saved successfully!
